In [2]:
import pandas as pd
import numpy as np
from pathlib import Path 
import os 
import matplotlib.pyplot as plt
import random

In [3]:
pathm = os.path.abspath(os.path.join(os.getcwd(), '..'))
pathm = os.path.join(pathm, "data", "raw", "menza_prices.csv")

canteen = pd.read_csv(pathm)

In [4]:
canteen

,date,meal_type,name,price
0,2026-04-19,Polévka,Kuřecí vývar s nudlemi,22.0
1,2026-04-19,Hlavní jídlo,"Kuřecí roláda plněná, dušená rýže",103.0
2,2026-04-19,Hlavní jídlo,"Segedínský guláš, houskový knedlík",95.0
3,2026-04-19,Hlavní jídlo,"Mac and Cheese, malý míchaný salát",99.0
4,2026-04-19,Hlavní jídlo,"Čočkový kotlík, chléb",74.0
5,2026-04-19,Hlavní jídlo,"Vepřová kotleta na grilu, dip, americké brambory",107.0
6,2026-04-20,Polévka,Dršťková polévka z hlívy ústřičné,22.0
7,2026-04-20,Hlavní jídlo,"Hovězí stoganoff, dušená rýže",115.0
8,2026-04-20,Hlavní jídlo,"Domácí pečená sekaná, bramborová kaše, okurka",102.0
9,2026-04-20,Hlavní jídlo,Kuskusový nákyp se zeleninou,87.0


In [5]:
canteen_clean=canteen[["meal_type","price"]]

In [6]:
canteen_stats = canteen_clean.groupby("meal_type").agg(
    avg_price=("price", "mean"),
    count=("price", "count")
).reset_index()

In [7]:
canteen["date"] = pd.to_datetime(canteen["date"])

In [8]:
main_meal_mean=canteen[canteen["meal_type"]=="Hlavní jídlo"]["price"].mean()
soup_mean=canteen[canteen["meal_type"]=="Polévka"]["price"].mean()

In [9]:
main_meal_mean
soup_mean

np.float64(22.0)

In [10]:
daily_cost=main_meal_mean+soup_mean
weekly_cost=daily_cost*5
monthly_cost=weekly_cost*4
monthly_cost

np.float64(2520.8)

In [14]:
canteen_costs=pd.DataFrame({"days_per_week":range(1,6),
                           "weeks_per_month":4})

canteen_costs["monthly_cost"]=(
    canteen_costs["days_per_week"]*canteen_costs["weeks_per_month"]*(main_meal_mean+soup_mean)
)

canteen_costs=canteen_costs.drop(columns="weeks_per_month")
canteen_costs

,days_per_week,monthly_cost
0,1,504.16
1,2,1008.32
2,3,1512.48
3,4,2016.64
4,5,2520.80


In [16]:
os.chdir("..")
canteen_costs.to_csv(
    "data/clean/canteen_costs.csv",
    index=False
)

In [12]:
#for the simulation
def calculate_monthly_menza_cost(df, days_per_week, include_soup=False):
    """
    Calculate estimated monthly canteen cost based on randomly chosen dining days.
    
    Parameters:
    df: pandas.DataFrame (with columns: date, meal_type,name,price)
    days_per_week: int (number of days user wants to eat at menza (1-5))
    include_soup: bool (with eachmeal, default: False)

    Returns:
    dict: COntains average daily cost, weekly cost, and estimated monthly cost

    """

    #validate input
    if not(1<= days_per_week <=5):
        raise ValueError("Choose between 1 and 5 days per week")
    
    #create cope, date column to datetime
    df=df.copy()
    df["date"]=pd.to_datetime(df["date"])
    
    #get unique dates in dataset
    unique_dates=sorted(df["date"].unique())
    
    if len(unique_dates)<days_per_week:
        raise ValueError(f"Dataset has {len(unique_dates)} days, but you want {days_per_week} days/week")
    
    #randomly select "days_per_week" days from the dataset
    selected_dates=random.sample(list(unique_dates),days_per_week)

    #filter main courses from selected dates
    selected_meals =df[(df["date"].isin(selected_dates)) &
                       (df["meal_type"]=="Hlavní jídlo")]
    
    #filter soups from selected dates
    soups=df[(df["date"].isin(selected_dates)) &
             (df["meal_type"]=="Polévka")]
    
    #calculate average price per day
    daily_costs=[]
    meal_details=[]

    for date in selected_dates:
        meals_on_date=selected_meals[selected_meals["date"]==date]

        if len(meals_on_date)>0:
            #randomly select one main course
            random_meal=meals_on_date.sample(1)
            meal_price=random_meal["price"].values[0]
            meal_name=random_meal["name"].values[0]

            day_cost=meal_price
            day_details=f"{meal_name}:{meal_price} CZK"

            #add soup if requested
            if include_soup:
                soups_on_date=soups[soups["date"]==date]
            
                if len(soups_on_date)>0:
                    soup_price=soups_on_date["price"].iloc[0]
                    soup_name=soups_on_date["name"].iloc[0]

                    day_cost+=soup_price
                    day_details+=f"+ {soup_name}:{soup_price} CZK"

            daily_costs.append(day_cost)
            meal_details.append({
                "date":date,
                "details":day_details,
                "cost":day_cost
            })

    #calculate statistics            
    average_daily_cost=sum(daily_costs)/len(daily_costs) if daily_costs else 0
    weekly_cost=average_daily_cost*days_per_week
    monthly_cost=weekly_cost*4

    return{
        "days_per_week":days_per_week,
        "include_soup":include_soup,
        "selected_dates":sorted(selected_dates),
        "meal_details":meal_details,
        "daily_costs":daily_costs,
        "average_daily_cost":round(average_daily_cost,2),
        "weekly_cost":round(weekly_cost,2),
        "monthly_cost":round(monthly_cost,2)
    }

    

In [13]:
calculate_monthly_menza_cost(canteen,5,include_soup=True)

{'days_per_week': 5,
 'include_soup': True,
 'selected_dates': [Timestamp('2026-04-19 00:00:00'),
  Timestamp('2026-04-20 00:00:00'),
  Timestamp('2026-04-21 00:00:00'),
  Timestamp('2026-04-22 00:00:00'),
  Timestamp('2026-04-23 00:00:00')],
 'meal_details': [{'date': Timestamp('2026-04-21 00:00:00'),
   'details': 'Ratatouille s červenými fazolemi, pitta chléb:118.0 CZK+ Zeleninová polévka s praženou krupicí:22.0 CZK',
   'cost': np.float64(140.0)},
  {'date': Timestamp('2026-04-23 00:00:00'),
   'details': 'Ovocné knedlíky s tvarohem, cukrem a jogurtovým přellivem:108.0 CZK+ Hrstková polévka:22.0 CZK',
   'cost': np.float64(130.0)},
  {'date': Timestamp('2026-04-22 00:00:00'),
   'details': 'VEGAN Smažený sýr gouda, vařené brambory, vegan tatarská omáčka:107.0 CZK+ Zelná polévka s bramborem:22.0 CZK',
   'cost': np.float64(129.0)},
  {'date': Timestamp('2026-04-19 00:00:00'),
   'details': 'Vepřová kotleta na grilu, dip, americké brambory:107.0 CZK+ Kuřecí vývar s nudlemi:22.0 CZK',

In [14]:
#multiple repetition - more robust simulation 
def simulate_monthly_cost(df, days_per_week, include_soup=False, n_simulations=1000):
    results=[]
    for k in range(n_simulations):
        r=calculate_monthly_menza_cost(df,days_per_week,include_soup)
        results.append(r["monthly_cost"])

    return {
        "mean_monthly_cost": round(sum(results)/len(results),2),
        "min_monthly_cost":round(min(results),2),
        "max_monthly_cost":round(max(results),2),
        "std_monthly_cost":np.std(results)
    }

In [15]:
simulate_monthly_cost(canteen,5,include_soup=True)

{'mean_monthly_cost': np.float64(2525.74),
 'min_monthly_cost': np.float64(2204.0),
 'max_monthly_cost': np.float64(2848.0),
 'std_monthly_cost': np.float64(107.43003490644503)}